# **VIII) Granger Causality Analysis**



1. [**Import Data**](#1-import--data)
2. [**Significance Formatting**](#2-significance)
3. [**Granger Causality Testing Loop**](#3-granger)
4. [**LaTeX Output**](#4-latex)

### **1) Import Data** <a id="1-import--data"></a>

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests

df = pd.read_csv("data/All_Data_Weekly_transformed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

target = "btc_ret"

stationary_predictors = [
    "btc_ret", "mp_exp_d", "news_sent", "policy_risk_gr",
    "sp500_ret", "brent_gr", "gold_ret", "hy_ret",
    "gpr_gr", "vix_gr", "dxy_gr", "emv",
    "claims_dlog", "eer_dlog", "ffr_d2", "infl5y_d",
    "gt_infl_d", "gt_recess", "gt_climate"
]

MAX_LAG = 6

### **2) Significance Formatting** <a id="2-significance"></a>

In [2]:
# Helpers Statistical Significance
def sig_stars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.01:
        return r"$^{***}$"
    if p < 0.05:
        return r"$^{**}$"
    if p < 0.10:
        return r"$^{*}$"
    return ""

def format_p(p: float) -> str:
    if pd.isna(p):
        return ""
    # typical econ formatting: 3 decimals + significance markers
    return f"{p:.3f}{sig_stars(p)}"

### **3) Granger Causality Testing Loop** <a id="3-granger"></a>

In [3]:
pvals = pd.DataFrame(index=stationary_predictors, columns=[f"lag_{l}" for l in range(1, MAX_LAG + 1)], dtype=float)

for x in stationary_predictors:
    # statsmodels expects a 2-col array-like: [y, x]
    data_pair = df[[target, x]].dropna()
    
    # Check for sufficient observations
    if len(data_pair) <= (MAX_LAG + 5):
        continue

    res = grangercausalitytests(data_pair, maxlag=MAX_LAG)
    
    for lag in range(1, MAX_LAG + 1):
        # Extract the p-value for the SSR F-test at each lag
        pvals.loc[x, f"lag_{lag}"] = res[lag][0]["ssr_ftest"][1]

# Apply formatting (stars and decimals)
pvals_formatted = pvals.map(format_p)

# Optional: nicer row label
pvals_formatted.index.name = "variable"
pvals_formatted.columns.name = "granger_lag"

display(pvals_formatted)


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.0000  , p=1.0000  , df_denom=542, df_num=1
ssr based chi2 test:   chi2=0.0000  , p=1.0000  , df=1
likelihood ratio test: chi2=-0.0000 , p=1.0000  , df=1
parameter F test:         F=1.8410  , p=0.1754  , df_denom=542, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=0.0000  , p=1.0000  , df_denom=540, df_num=2
ssr based chi2 test:   chi2=0.0000  , p=1.0000  , df=2
likelihood ratio test: chi2=-0.0000 , p=1.0000  , df=2
parameter F test:         F=1.0318  , p=0.3571  , df_denom=540, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=0.0000  , p=1.0000  , df_denom=538, df_num=3
ssr based chi2 test:   chi2=0.0000  , p=1.0000  , df=3
likelihood ratio test: chi2=-0.0000 , p=1.0000  , df=3
parameter F test:         F=1.1144  , p=0.3427  , df_denom=538, df_num=3

Granger Causality
number of lags (no zero) 4
ssr based F test:         F=0.0000  , p=1.0000  

granger_lag,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6
variable,,,,,,
btc_ret,1.000,1.000,1.000,1.000,1.000,1.000
mp_exp_d,0.225,0.108,0.069$^{*}$,0.092$^{*}$,0.089$^{*}$,0.139
news_sent,0.996,0.942,0.897,0.905,0.750,0.855
policy_risk_gr,0.196,0.400,0.350,0.462,0.562,0.583
sp500_ret,0.236,0.468,0.407,0.576,0.734,0.704
brent_gr,0.998,0.400,0.561,0.469,0.345,0.435
gold_ret,0.077$^{*}$,0.042$^{**}$,0.080$^{*}$,0.085$^{*}$,0.076$^{*}$,0.124
hy_ret,0.140,0.421,0.557,0.730,0.848,0.892
gpr_gr,0.893,0.056$^{*}$,0.109,0.165,0.250,0.310


### **4) LaTeX Output** <a id="4-latex"></a>

In [4]:
# Mapping
names = {
    "btc_ret": "Btc", "mp_exp_d": "MPE", 
    "news_sent": "NewsSent", "policy_risk_gr": "PolUncert",
    "sp500_ret": "SP500", "brent_gr": "Brent", 
    "gold_ret": "Gold", "hy_ret": "HighYield",
    "gpr_gr": "GeopolRisk", "vix_gr": "VIX", "dxy_gr": "USDollar",
    "emv": "Infect",
    "claims_dlog": "JoblessClaim", "eer_dlog": "ExchRate", 
    "ffr_d2": "FFR", "infl5y_d": "5yInflExp",
    "gt_infl_d": "GgleInfl", "gt_recess": "GgleReces", "gt_climate": "GgleClimate",
}

def latex_texttt(s: str) -> str:
    return rf"\texttt{{{s.replace('_', r'\_')}}}"

# replace lag_1, lag_2 strings with just the numbers
pvals_display = pvals_formatted.rename(index=names)
pvals_display.columns = [c.replace("lag_", "") for c in pvals_display.columns]

# Remove LaTeX stars for clean notebook viewing
pvals_plain = pvals_display.replace(r"\$\^\{([^}]*)\}\$", r"\1", regex=True)
display(pvals_plain)

pvals_tex = pvals_formatted.copy()

# Index: Academic Name + Code Name in texttt
pvals_tex.index = [f"{names[v]}" for v in pvals_tex.index]

# Columns: Just the lag numbers to save horizontal space
pvals_tex.columns = [c.replace("lag_", "") for c in pvals_tex.columns]

# LaTeX italics formatting
pvals_tex.index = [f"\\textit{{{name}}}" for name in pvals_tex.index]

latex_table = pvals_tex.to_latex(
    escape=False, # Crucial for stars and texttt
    na_rep="",
    column_format="l" + "c" * pvals_tex.shape[1],
    caption="Granger Causality Tests: p-values from SSR F-tests (Dependent Variable: BTC returns)",
    label="tab:granger_btc",
)

print(latex_table)

,1,2,3,4,5,6
variable,,,,,,
Btc,1.000,1.000,1.000,1.000,1.000,1.000
MPE,0.225,0.108,0.069*,0.092*,0.089*,0.139
NewsSent,0.996,0.942,0.897,0.905,0.750,0.855
PolUncert,0.196,0.400,0.350,0.462,0.562,0.583
SP500,0.236,0.468,0.407,0.576,0.734,0.704
Brent,0.998,0.400,0.561,0.469,0.345,0.435
Gold,0.077*,0.042**,0.080*,0.085*,0.076*,0.124
HighYield,0.140,0.421,0.557,0.730,0.848,0.892
GeopolRisk,0.893,0.056*,0.109,0.165,0.250,0.310


\begin{table}
\caption{Granger Causality Tests: p-values from SSR F-tests (Dependent Variable: BTC returns)}
\label{tab:granger_btc}
\begin{tabular}{lcccccc}
\toprule
 & 1 & 2 & 3 & 4 & 5 & 6 \\
\midrule
\textit{Btc} & 1.000 & 1.000 & 1.000 & 1.000 & 1.000 & 1.000 \\
\textit{MPE} & 0.225 & 0.108 & 0.069$^{*}$ & 0.092$^{*}$ & 0.089$^{*}$ & 0.139 \\
\textit{NewsSent} & 0.996 & 0.942 & 0.897 & 0.905 & 0.750 & 0.855 \\
\textit{PolUncert} & 0.196 & 0.400 & 0.350 & 0.462 & 0.562 & 0.583 \\
\textit{SP500} & 0.236 & 0.468 & 0.407 & 0.576 & 0.734 & 0.704 \\
\textit{Brent} & 0.998 & 0.400 & 0.561 & 0.469 & 0.345 & 0.435 \\
\textit{Gold} & 0.077$^{*}$ & 0.042$^{**}$ & 0.080$^{*}$ & 0.085$^{*}$ & 0.076$^{*}$ & 0.124 \\
\textit{HighYield} & 0.140 & 0.421 & 0.557 & 0.730 & 0.848 & 0.892 \\
\textit{GeopolRisk} & 0.893 & 0.056$^{*}$ & 0.109 & 0.165 & 0.250 & 0.310 \\
\textit{VIX} & 0.464 & 0.581 & 0.611 & 0.768 & 0.899 & 0.923 \\
\textit{USDollar} & 0.548 & 0.585 & 0.801 & 0.854 & 0.940 & 0.863 \\
\te